# Pedagogical Assessment Pipeline - Interactive Notebook

This notebook demonstrates the complete pipeline for:
1. Solving math problems and generating answer keys
2. Evaluating tutor responses for mistake identification
3. Creating train/validation/test splits

## Setup

In [ ]:
# Install required packages
!pip install openai anthropic tqdm -q

In [ ]:
import json
import os
from pathlib import Path
import sys

# Import the pipeline
from pedagogical_assessment_pipeline import (
    PedagogicalAssessmentPipeline,
    MathProblemSolver,
    MistakeIdentificationEvaluator
)

In [ ]:
# Set your API key here (or use environment variable)
# Option 1: Set environment variable before running notebook
# Option 2: Uncomment and set here (NOT recommended for shared notebooks)

# os.environ["OPENAI_API_KEY"] = "your-key-here"
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"

# Check which API keys are available
has_openai = "OPENAI_API_KEY" in os.environ
has_anthropic = "ANTHROPIC_API_KEY" in os.environ

print(f"OpenAI API key set: {has_openai}")
print(f"Anthropic API key set: {has_anthropic}")

if not has_openai and not has_anthropic:
    print("\n⚠️  WARNING: No API key found!")
    print("Please set OPENAI_API_KEY or ANTHROPIC_API_KEY")

## Configuration

In [ ]:
# Configuration
DATA_DIR = "/DATA/cs24resch11011/repos/pedagogical-assessment/data"
OUTPUT_DIR = "/DATA/cs24resch11011/repos/pedagogical-assessment/devendra/mistake_identification/llm_approach/notebook_output"

# Choose provider and model
PROVIDER = "openai" if has_openai else "anthropic"
SOLVER_MODEL = "gpt-4o" if PROVIDER == "openai" else "claude-3-5-sonnet-20241022"
EVALUATOR_MODEL = SOLVER_MODEL

# For testing, limit the number of samples
LIMIT = 3  # Set to None to process all
RATE_LIMIT_DELAY = 1.0  # seconds

print(f"Provider: {PROVIDER}")
print(f"Solver Model: {SOLVER_MODEL}")
print(f"Evaluator Model: {EVALUATOR_MODEL}")
print(f"Sample Limit: {LIMIT}")
print(f"Output Directory: {OUTPUT_DIR}")

## Load and Explore Data

In [ ]:
# Load the training set
with open(f"{DATA_DIR}/trainset.json", 'r') as f:
    trainset = json.load(f)

print(f"Total conversations in trainset: {len(trainset)}")
print(f"\nSample conversation structure:")
print(json.dumps(trainset[0], indent=2)[:1000] + "...")

In [ ]:
# Examine a single conversation
sample = trainset[0]
print("Conversation ID:", sample["conversation_id"])
print("\nConversation History:")
print(sample["conversation_history"][:500] + "...")
print("\nTutor Responses:")
for tutor_name, response_data in sample["tutor_responses"].items():
    if response_data:
        print(f"\n{tutor_name}:")
        print(f"  Response: {response_data['response'][:100]}...")
        print(f"  Annotation: {response_data['annotation']}")

## Step 1: Test Math Problem Solver on Single Example

In [ ]:
# Initialize solver
solver = MathProblemSolver(model=SOLVER_MODEL, provider=PROVIDER)

# Extract and solve a problem
sample_conversation = trainset[0]
problem = solver.extract_math_problem(sample_conversation["conversation_history"])

print("Extracted Problem:")
print(problem)
print("\n" + "="*80 + "\n")

if problem:
    solution = solver.solve_problem(problem)
    print("Solution:")
    print(json.dumps(solution, indent=2))
else:
    print("No problem found in this conversation.")

## Step 2: Test Evaluator on Single Response

In [ ]:
# Initialize evaluator
evaluator = MistakeIdentificationEvaluator(model=EVALUATOR_MODEL, provider=PROVIDER)

# Get a tutor response to evaluate
sample_tutor = list(sample_conversation["tutor_responses"].items())[0]
tutor_name, response_data = sample_tutor

if response_data:
    print(f"Evaluating response from: {tutor_name}")
    print(f"\nResponse: {response_data['response']}")
    print(f"\nOriginal Annotation: {response_data['annotation']}")
    print("\n" + "="*80 + "\n")
    
    evaluation = evaluator.evaluate_response(
        sample_conversation["conversation_history"],
        solution["answer"],
        response_data["response"],
        response_data["annotation"]
    )
    
    print("Evaluation Result:")
    print(json.dumps(evaluation, indent=2))

## Step 3: Run Full Pipeline on Sample

In [ ]:
# Initialize pipeline
pipeline = PedagogicalAssessmentPipeline(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    solver_model=SOLVER_MODEL,
    evaluator_model=EVALUATOR_MODEL,
    provider=PROVIDER
)

In [ ]:
# Run Step 1: Add answer keys
print("Step 1: Solving problems and adding answer keys...")
enriched_trainset = pipeline.process_trainset_with_answers(
    limit=LIMIT,
    rate_limit_delay=RATE_LIMIT_DELAY
)
pipeline.save_dataset(enriched_trainset, "trainset_with_answers.json")
print(f"✓ Processed {len(enriched_trainset)} conversations")

In [ ]:
# Examine enriched data
print("Sample enriched conversation:")
enriched_sample = enriched_trainset[0]
print(f"\nConversation ID: {enriched_sample['conversation_id']}")
if 'math_problem' in enriched_sample:
    print(f"\nMath Problem: {enriched_sample['math_problem'][:200]}...")
if 'answer_key' in enriched_sample:
    print(f"\nAnswer Key:")
    print(json.dumps(enriched_sample['answer_key'], indent=2))

In [ ]:
# Run Step 2: Evaluate responses
print("Step 2: Evaluating tutor responses...")
evaluated_data = pipeline.evaluate_all_responses(
    enriched_trainset,
    limit=LIMIT,
    rate_limit_delay=RATE_LIMIT_DELAY
)
pipeline.save_dataset(evaluated_data, "trainset_fully_evaluated.json")
print(f"✓ Evaluated responses for {len(evaluated_data)} conversations")

In [ ]:
# Examine evaluated data
print("Sample evaluated conversation:")
eval_sample = evaluated_data[0]
print(f"\nConversation ID: {eval_sample['conversation_id']}")

if 'tutor_responses' in eval_sample:
    for tutor_name, response_data in list(eval_sample['tutor_responses'].items())[:2]:
        if response_data and 'evaluation' in response_data:
            print(f"\n{tutor_name}:")
            print(f"  Original Annotation:")
            print(f"    MI: {response_data['annotation'].get('Mistake_Identification')}")
            print(f"    PG: {response_data['annotation'].get('Providing_Guidance')}")
            print(f"  Predicted:")
            print(f"    MI: {response_data['evaluation']['predicted_mistake_identification']}")
            print(f"    PG: {response_data['evaluation']['predicted_providing_guidance']}")
            print(f"  Confidence: {response_data['evaluation']['confidence']:.2f}")
            print(f"  Agrees: {response_data['evaluation']['agrees_with_human']}")

In [ ]:
# Run Step 3: Generate statistics
print("Step 3: Generating statistics...")
stats = pipeline.generate_statistics(evaluated_data)
pipeline.save_dataset([stats], "dataset_statistics.json")

print("\nDataset Statistics:")
print(json.dumps(stats, indent=2))

In [ ]:
# Run Step 4: Create splits
print("Step 4: Creating train/validation/test splits...")
train_set, val_set, test_set = pipeline.create_train_val_test_splits(evaluated_data)

pipeline.save_dataset(train_set, "train_split.json")
pipeline.save_dataset(val_set, "validation_split.json")
pipeline.save_dataset(test_set, "test_split.json")

print(f"\nSplit sizes:")
print(f"  Train: {len(train_set)}")
print(f"  Validation: {len(val_set)}")
print(f"  Test: {len(test_set)}")

## Visualize Results

In [ ]:
# Install visualization libraries if needed
!pip install matplotlib seaborn pandas -q

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Plot mistake identification distribution
mi_dist = stats['mistake_identification_distribution']
pg_dist = stats['providing_guidance_distribution']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mistake Identification
axes[0].bar(mi_dist.keys(), mi_dist.values(), color=['green', 'red', 'orange'])
axes[0].set_title('Mistake Identification Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Category')

# Providing Guidance
axes[1].bar(pg_dist.keys(), pg_dist.values(), color=['green', 'red', 'orange'])
axes[1].set_title('Providing Guidance Distribution', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Category')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/distribution_plots.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {OUTPUT_DIR}/distribution_plots.png")

In [ ]:
# Agreement with human annotations
agreement = stats['agreement_with_human']
labels = ['Agrees', 'Disagrees']
values = [agreement['yes'], agreement['no']]
colors = ['green', 'red']

plt.figure(figsize=(8, 8))
plt.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
plt.title('Agreement with Human Annotations', fontsize=14, fontweight='bold')
plt.savefig(f"{OUTPUT_DIR}/agreement_pie.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {OUTPUT_DIR}/agreement_pie.png")

## Analyze Individual Responses

In [ ]:
# Create a dataframe of all evaluations
eval_records = []

for conv in evaluated_data:
    conv_id = conv['conversation_id']
    if 'tutor_responses' in conv:
        for tutor_name, response_data in conv['tutor_responses'].items():
            if response_data and 'evaluation' in response_data:
                eval_records.append({
                    'conversation_id': conv_id,
                    'tutor': tutor_name,
                    'human_mi': response_data['annotation'].get('Mistake_Identification', 'Unknown'),
                    'human_pg': response_data['annotation'].get('Providing_Guidance', 'Unknown'),
                    'pred_mi': response_data['evaluation']['predicted_mistake_identification'],
                    'pred_pg': response_data['evaluation']['predicted_providing_guidance'],
                    'confidence': response_data['evaluation']['confidence'],
                    'agrees': response_data['evaluation']['agrees_with_human']
                })

df = pd.DataFrame(eval_records)
print(f"Total evaluations: {len(df)}")
df.head(10)

In [ ]:
# Summary by tutor
print("Summary by Tutor:")
print(df.groupby('tutor').agg({
    'confidence': 'mean',
    'agrees': 'sum'
}).round(3))

In [ ]:
# Cases where model disagrees with human
print("Cases where model disagrees with human annotation:")
disagreements = df[df['agrees'] == False]
print(f"\nTotal disagreements: {len(disagreements)}")
disagreements[['conversation_id', 'tutor', 'human_mi', 'pred_mi', 'human_pg', 'pred_pg', 'confidence']]

## Export Summary Report

In [ ]:
# Create a summary report
report = f"""# Pedagogical Assessment Pipeline - Summary Report

## Configuration
- Provider: {PROVIDER}
- Model: {SOLVER_MODEL}
- Samples Processed: {len(evaluated_data)}

## Results

### Dataset Statistics
- Total Conversations: {stats['total_conversations']}
- Problems with Answers: {stats['problems_with_answers']}
- Total Responses: {stats['total_responses']}
- Evaluations Complete: {stats['evaluations_complete']}

### Agreement with Human Annotations
- Agree: {stats['agreement_with_human']['yes']}
- Disagree: {stats['agreement_with_human']['no']}
- Agreement Rate: {stats['agreement_with_human']['yes'] / (stats['agreement_with_human']['yes'] + stats['agreement_with_human']['no']) * 100:.1f}%

### Mistake Identification Distribution
- Yes: {mi_dist['Yes']}
- No: {mi_dist['No']}
- To some extent: {mi_dist['To some extent']}

### Providing Guidance Distribution
- Yes: {pg_dist['Yes']}
- No: {pg_dist['No']}
- To some extent: {pg_dist['To some extent']}

## Files Generated
- trainset_with_answers.json
- trainset_fully_evaluated.json
- dataset_statistics.json
- train_split.json
- validation_split.json
- test_split.json
- distribution_plots.png
- agreement_pie.png
"""

with open(f"{OUTPUT_DIR}/summary_report.md", 'w') as f:
    f.write(report)

print(report)
print(f"\nReport saved to: {OUTPUT_DIR}/summary_report.md")

## Next Steps

1. **Scale Up**: Remove the `LIMIT` parameter to process all conversations
2. **Train Models**: Use the generated splits to train mistake identification models
3. **Fine-tune**: Adjust prompts and models based on disagreements
4. **Validate**: Use validation set to tune hyperparameters
5. **Test**: Final evaluation on test set

## Running on Full Dataset

To process the entire dataset, either:
1. Set `LIMIT = None` in this notebook (may take hours)
2. Use the command-line script: `python pedagogical_assessment_pipeline.py`
3. Use the test script for a quick test: `python test_pipeline.py`